### Feature Engineering - Predicting Breast Tumor Diagnosis with Machine Learning

**Breast Cancer Wisconsin Diagnostic Dataset**<br>
Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). Breast Cancer Wisconsin (Diagnostic) [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B.

#### Variables
1) ID number
2) Diagnosis (M = malignant, B = benign)

3-32)
Ten real-valued features are computed for each cell nucleus:

	a) radius (mean of distances from center to points on the perimeter)
	b) texture (standard deviation of gray-scale values)
	c) perimeter
	d) area
	e) smoothness (local variation in radius lengths)
	f) compactness (perimeter^2 / area - 1.0)
	g) concavity (severity of concave portions of the contour)
	h) concave points (number of concave portions of the contour)
	i) symmetry 
	j) fractal dimension ("coastline approximation" - 1)

Go to [Kaggle](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data) for variable definitions

### Considerations for Feature Selection
***Excluded measurements***
- `area`, `perimeter`, and `radius` are all highly correlated with each other (makes sense). So I am selecting ONE of these as our tumor size variable. I've decided to go with `radius` because it appears to be the most independent from the other variables. 
- Similarly, `concave_points` is strongly correlated with the 3 size variables because bigger tumor -> more space for concave points to appear. I'm planning to exclude this metric. However, I do want some measure of concavity, so I'll keep `concavity` which is just moderately correlated with `radius` (r^2 = 0.68 vs 0.82)

***Included measurements***
- I will include `radius`,`texture`,`smoothness`,`compactness`,`concavity`,`symmetry`, and `fractal dimension`. 
- This dataset contains the mean, standard error, and worst value for each measurement. I will loop through different variable combinations to find what works best while avoiding linked variables such as `radius_mean` and `radius_se`.

### Imports

In [84]:
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import StandardScaler

import itertools
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score  # Change to roc_auc_score if needed


### Load Data

In [85]:
# Load the dataset
df = pd.read_csv('raw_tumor_feature_data.csv')
df.head(3)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [86]:
# Subset df to the measurements we've decided to keep 
col_to_keep = ['diagnosis', 'radius_mean', 'texture_mean','smoothness_mean', 'compactness_mean', \
               'concavity_mean','symmetry_mean', 'fractal_dimension_mean',\
                'radius_se', 'texture_se', 'smoothness_se','compactness_se', 'concavity_se',\
                    'symmetry_se','fractal_dimension_se', 'radius_worst', 'texture_worst',\
                        'smoothness_worst','compactness_worst', 'concavity_worst',\
                            'symmetry_worst', 'fractal_dimension_worst']

df_subset = df[col_to_keep]
df_subset.head(3)

,diagnosis,radius_mean,texture_mean,smoothness_mean,compactness_mean,concavity_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,smoothness_se,compactness_se,concavity_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,smoothness_worst,compactness_worst,concavity_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,0.11840,0.27760,0.3001,0.2419,0.07871,1.0950,0.9053,0.006399,0.04904,0.05373,0.03003,0.006193,25.38,17.33,0.1622,0.6656,0.7119,0.4601,0.11890
1,M,20.57,17.77,0.08474,0.07864,0.0869,0.1812,0.05667,0.5435,0.7339,0.005225,0.01308,0.01860,0.01389,0.003532,24.99,23.41,0.1238,0.1866,0.2416,0.2750,0.08902
2,M,19.69,21.25,0.10960,0.15990,0.1974,0.2069,0.05999,0.7456,0.7869,0.006150,0.04006,0.03832,0.02250,0.004571,23.57,25.53,0.1444,0.4245,0.4504,0.3613,0.08758


In [87]:
df_subset.diagnosis.value_counts()

diagnosis
B    357
M    212
Name: count, dtype: int64

In [88]:
# Split into X and y
y = df_subset['diagnosis']
X = df_subset.drop('diagnosis',axis=1)
print(y.shape, X.shape)

(569,) (569, 21)


In [89]:
# Modify y so M=1 and B=0
y = y.replace({'M':1,'B':0})
y.value_counts()

/var/folders/cq/gp4mmh_532d1t3573lxjzj5w0000gp/T/ipykernel_64049/2697745266.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({'M':1,'B':0})


diagnosis
0    357
1    212
Name: count, dtype: int64

In [90]:
X.columns

Index(['radius_mean', 'texture_mean', 'smoothness_mean', 'compactness_mean',
       'concavity_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'smoothness_se', 'compactness_se',
       'concavity_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst',
       'texture_worst', 'smoothness_worst', 'compactness_worst',
       'concavity_worst', 'symmetry_worst', 'fractal_dimension_worst'],
      dtype='object')

### Attempt 1: different combinations of the 3 feature groups

In [ ]:
# # Make different combinations of the feature groups so each dataframe has one radius, 
# # one texture, one smoothness, and so on

# # Extract all column names
# columns = X.columns

# # Group columns by feature type prefix (radius, texture, etc.)
# feature_groups = defaultdict(list)
# for col in columns:
#     feature_type = col.split('_')[0]
#     feature_groups[feature_type].append(col)

# # Sort and prepare feature group list
# sorted_feature_groups = [feature_groups[feature] for feature in sorted(feature_groups.keys())]

# # Track the best combo
# best_score = 0
# best_combo = None

# for combo in itertools.product(*sorted_feature_groups):
#     X_subset = X[list(combo)]
    
#     # Train-test split
#     X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.2, random_state=42)
    
#     # Fit Random Forest
#     clf = RandomForestClassifier(random_state=42)
#     clf.fit(X_train, y_train)
    
#     # Predict and evaluate
#     y_pred = clf.predict(X_test)
#     acc = accuracy_score(y_test, y_pred)
    
#     # Track best
#     if acc > best_score:
#         best_score = acc
#         best_combo = combo

#     # print(f"Combo: {combo} | Accuracy: {acc:.4f}")

# print("\nBest combo:")
# print(best_combo)
# print(f"Best accuracy: {best_score:.4f}")

KeyboardInterrupt: 

#### Notes on Attempt 1
I interrupted the program because it's going to take way too much computational power. What if we use random choice to pick 5-10 combinations? If the model accuracy doesn't vary much between combos, we don't need to test every possible combo.

### Attempt 2: random choice

In [93]:
# How many random combinations to test
n_trials = 7

results = []

for i in range(n_trials):
    # Pick one random column from each group
    random_combo = [random.choice(group) for group in sorted_feature_groups]
    X_subset = X[random_combo]
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.2, random_state=42)
    
    # Train Random Forest
    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_train, y_train)
    
    # Evaluate
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    results.append((random_combo, acc))
    print(f"Trial {i+1}: Combo = {random_combo} | Accuracy = {acc:.4f}")

# Sort and show best combo
best_combo, best_score = max(results, key=lambda x: x[1])
print("\nBest combo:")
print(best_combo)
print(f"Best accuracy: {best_score:.4f}")

Trial 1: Combo = ['compactness_se', 'concavity_mean', 'fractal_dimension_se', 'radius_mean', 'smoothness_worst', 'symmetry_worst', 'texture_mean'] | Accuracy = 0.9825
Trial 2: Combo = ['compactness_mean', 'concavity_se', 'fractal_dimension_mean', 'radius_se', 'smoothness_worst', 'symmetry_se', 'texture_mean'] | Accuracy = 0.9386
Trial 3: Combo = ['compactness_se', 'concavity_se', 'fractal_dimension_worst', 'radius_worst', 'smoothness_mean', 'symmetry_worst', 'texture_worst'] | Accuracy = 0.9561
Trial 4: Combo = ['compactness_mean', 'concavity_worst', 'fractal_dimension_mean', 'radius_worst', 'smoothness_se', 'symmetry_worst', 'texture_worst'] | Accuracy = 0.9474
Trial 5: Combo = ['compactness_worst', 'concavity_mean', 'fractal_dimension_se', 'radius_se', 'smoothness_se', 'symmetry_se', 'texture_worst'] | Accuracy = 0.9474
Trial 6: Combo = ['compactness_se', 'concavity_worst', 'fractal_dimension_se', 'radius_se', 'smoothness_se', 'symmetry_mean', 'texture_mean'] | Accuracy = 0.9561
Tria

#### Notes on Attempt 2
Best combo:
['compactness_se', 'concavity_mean', 'fractal_dimension_se', 'radius_mean', 'smoothness_worst', 'symmetry_worst', 'texture_mean']
Best accuracy: 0.9825

This was faster, but there does seem to be some variation in the accuracy scores, so randomly picking a small number of combinations could be missing out on the best model. Don't get me wrong, this accuracy score is high, but I'm trying to build a model that is decisively stronger than the MSM Tree in the original paper. I went back to the paper and noticed that the authors only used 3 variables in their final model: `texture_mean`, `area_worst`, and `smoothness_worst` no `_se` variables. Let's see what happens if I test all combos without `_se`

### Attempt 3: no SE variables

In [94]:
# Try with no SE variables
col_to_keep = ['diagnosis', 'radius_mean', 'texture_mean','smoothness_mean', 'compactness_mean', \
               'concavity_mean','symmetry_mean', 'fractal_dimension_mean',\
                'radius_worst', 'texture_worst','smoothness_worst','compactness_worst', 
                'concavity_worst','symmetry_worst', 'fractal_dimension_worst']

df_subset = df[col_to_keep]

# Split into X and y
y = df_subset['diagnosis']
X = df_subset.drop('diagnosis',axis=1)

# Modify y so M=1 and B=0
y = y.replace({'M':1,'B':0})

/var/folders/cq/gp4mmh_532d1t3573lxjzj5w0000gp/T/ipykernel_64049/1154919762.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({'M':1,'B':0})


In [95]:
# Make different combinations of the feature groups so each dataframe has one radius, 
# one texture, one smoothness, and so on

# Extract all column names
columns = X.columns

# Group columns by feature type prefix (radius, texture, etc.)
feature_groups = defaultdict(list)
for col in columns:
    feature_type = col.split('_')[0]
    feature_groups[feature_type].append(col)

# Sort and prepare feature group list
sorted_feature_groups = [feature_groups[feature] for feature in sorted(feature_groups.keys())]

# Track the best combo
best_score = 0
best_combo = None

for combo in itertools.product(*sorted_feature_groups):
    X_subset = X[list(combo)]
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.2, random_state=42)
    
    # Fit Random Forest
    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    # Track best
    if acc > best_score:
        best_score = acc
        best_combo = combo

    # print(f"Combo: {combo} | Accuracy: {acc:.4f}")

print("\nBest combo:")
print(best_combo)
print(f"Best accuracy: {best_score:.4f}")


Best combo:
('compactness_mean', 'concavity_mean', 'fractal_dimension_mean', 'radius_mean', 'smoothness_worst', 'symmetry_mean', 'texture_mean')
Best accuracy: 0.9825


#### Notes on attempt 3
Best combo:
('compactness_mean', 'concavity_mean', 'fractal_dimension_mean', 'radius_mean', 'smoothness_worst', 'symmetry_mean', 'texture_mean')
Best accuracy: 0.9825

This got the same accuracy as the best random combination. Interestingly, 2 of the 3 variables from the paper's final model are in this combination, `smoothness_worst` and `texture_mean`. Let's see what happens if I re-run with area instead of radius. Will the accuracy go up? Will `area_worst` be included?

### Attempt 4: area instead of radius

In [96]:
# Try with area instead of radius variables
col_to_keep = ['diagnosis', 'area_mean', 'texture_mean','smoothness_mean', 'compactness_mean', \
               'concavity_mean','symmetry_mean', 'fractal_dimension_mean',\
                'area_worst', 'texture_worst','smoothness_worst','compactness_worst', 
                'concavity_worst','symmetry_worst', 'fractal_dimension_worst']

df_subset = df[col_to_keep]

# Split into X and y
y = df_subset['diagnosis']
X = df_subset.drop('diagnosis',axis=1)

# Modify y so M=1 and B=0
y = y.replace({'M':1,'B':0})

/var/folders/cq/gp4mmh_532d1t3573lxjzj5w0000gp/T/ipykernel_64049/459706396.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({'M':1,'B':0})


In [97]:
# Make different combinations of the feature groups so each dataframe has one radius, 
# one texture, one smoothness, and so on

# Extract all column names
columns = X.columns

# Group columns by feature type prefix (radius, texture, etc.)
feature_groups = defaultdict(list)
for col in columns:
    feature_type = col.split('_')[0]
    feature_groups[feature_type].append(col)

# Sort and prepare feature group list
sorted_feature_groups = [feature_groups[feature] for feature in sorted(feature_groups.keys())]

# Track the best combo
best_score = 0
best_combo = None

for combo in itertools.product(*sorted_feature_groups):
    X_subset = X[list(combo)]
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.2, random_state=42)
    
    # Fit Random Forest
    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    # Track best
    if acc > best_score:
        best_score = acc
        best_combo = combo

    # print(f"Combo: {combo} | Accuracy: {acc:.4f}")

print("\nBest combo:")
print(best_combo)
print(f"Best accuracy: {best_score:.4f}")


Best combo:
('area_worst', 'compactness_mean', 'concavity_mean', 'fractal_dimension_mean', 'smoothness_worst', 'symmetry_mean', 'texture_mean')
Best accuracy: 0.9912


#### Notes on attempt 4
Best combo:
('area_worst', 'compactness_mean', 'concavity_mean', 'fractal_dimension_mean', 'smoothness_worst', 'symmetry_mean', 'texture_mean')
Best accuracy: 0.9912

Sure enough, `area_worst` beat out `area_mean` and the accuracy score is the highest yet! We can probably take this combo forward to the final steps.